# Supplementary Reproducible Analysis Notebook

This notebook aggregate findings reported in the paper:

1. Validation of the chat-session collection pipeline against the prior conversational-programming package.
2. RQ1 analyses on AI use, chat purposes, and chat-associated commits.
3. RQ2 analyses on repository dynamics before and after AI adoption.
4. RQ3 survey summaries.


In [1]:
from pathlib import Path
import json
import re
import zipfile

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
if ROOT.name != "vibe-coding":
    # Allows execution from the paper/ directory as well.
    if (ROOT.parent / "outputs").exists():
        ROOT = ROOT.parent
OUT = ROOT / "outputs"
PAPER = ROOT / "paper"

def require(path):
    path = ROOT / path if not isinstance(path, Path) else path
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def read_csv(path, **kwargs):
    return pd.read_csv(require(path), **kwargs)

def pct(x, digits=1):
    return round(float(x) * 100, digits)

def pct_already(x, digits=1):
    return round(float(x), digits)

def round_df(df, digits=3):
    out = df.copy()
    for col in out.select_dtypes(include=[np.number]).columns:
        out[col] = out[col].round(digits)
    return out

SENSITIVE_COL_PAT = re.compile(r"(email|e-mail|address|location|school|institution|raw_text|message_text|content|body)$", re.I)

def assert_no_display_sensitive_columns(df, context="dataframe"):
    bad = [c for c in df.columns if SENSITIVE_COL_PAT.search(str(c))]
    if bad:
        raise AssertionError(f"{context} contains sensitive columns that should not be displayed: {bad}")

def show(df, digits=3):
    assert_no_display_sensitive_columns(df)
    display(round_df(df, digits))

## 1. Cohorts And Output Files

- Full analysis cohort: 1,240 repositories with complete accessible histories.
- Main pre/post comparison cohort: 608 repositories whose first observed AI chat occurred at or after GitHub publication.
- Older validation cohort: 114 repositories created before 2025.

In [2]:
cohort_files = {
    "complete_history_filtered": "outputs/toy_homework_audit/complete_history_repos_excluding_strict_name_toy_homework_1240.csv",
    "main_comparison": "outputs/toy_homework_audit/main_comparison_chat_after_start_excluding_toy_homework_608.csv",
    "older_validation": "outputs/toy_homework_audit/older_pre2025_excluding_toy_homework_114.csv",
}

cohort_summary = []
for label, rel in cohort_files.items():
    df = read_csv(rel)
    cohort_summary.append({"cohort_file": label, "rows": len(df), "columns": len(df.columns)})
cohort_summary = pd.DataFrame(cohort_summary)
show(cohort_summary, 0)

assert int(cohort_summary.loc[cohort_summary.cohort_file == "complete_history_filtered", "rows"].iloc[0]) == 1240
assert int(cohort_summary.loc[cohort_summary.cohort_file == "main_comparison", "rows"].iloc[0]) == 608
assert int(cohort_summary.loc[cohort_summary.cohort_file == "older_validation", "rows"].iloc[0]) == 114

,cohort_file,rows,columns
0,complete_history_filtered,1240,13
1,main_comparison,608,13
2,older_validation,114,13


## 1.1 Analysis Provenance

In [3]:
provenance = pd.DataFrame([
    {
        "paper_section": "Cohort construction / toy-homework filtering",
        "local_script_or_builder": "paper/supplementary_data_analysis.py; py_script_analysis/*toy_homework* audit scripts",
        "aggregate_outputs_used": "outputs/toy_homework_audit/*.csv",
    },
    {
        "paper_section": "Data collection validation for chat sessions",
        "local_script_or_builder": "vibe-coding-scraper-main.zip: scrape.ipynb, parse.py, parse_cli.py",
        "aggregate_outputs_used": "outputs/chat_label_correction_1240/combined_seven_category_summary_1240.csv",
    },
    {
        "paper_section": "RQ1-1 AI-related repository activity",
        "local_script_or_builder": "paper/supplementary_data_analysis.py; py_script_analysis/ai_usage_chat_development_analysis.py",
        "aggregate_outputs_used": "outputs/repo_ai_adoption_filtered_1240/; outputs/ai_related_commit_share_extensions/",
    },
    {
        "paper_section": "RQ1-2 chat purpose and associated commits",
        "local_script_or_builder": "py_script_analysis/collect_missing_paper_label_batches.py; py_script_analysis/chat_purpose_commit_scope_audit.py",
        "aggregate_outputs_used": "outputs/chat_label_correction_1240/; outputs/chat_label_correction_1240/chat_associated_commit_recalc/",
    },
    {
        "paper_section": "RQ2-1 commit activity",
        "local_script_or_builder": "paper/supplementary_data_analysis.py; relative-month commit activity builders",
        "aggregate_outputs_used": "outputs/relative_month_commit_activity_filtered_1240/",
    },
    {
        "paper_section": "RQ2-2 quality and maintenance signals",
        "local_script_or_builder": "build_rq_quality_maintenance_monthly_notebook.py; build_rq_maintenance_followup_interval_notebook.py",
        "aggregate_outputs_used": "outputs/rq_quality_maintenance_filtered_1240/",
    },
    {
        "paper_section": "RQ2-3 issues and pull requests",
        "local_script_or_builder": "py_script_analysis/issue_pr_resolution_activity.py; issue/PR recalculation scripts",
        "aggregate_outputs_used": "outputs/rq_issue_pr_filtered_1240/",
    },
    {
        "paper_section": "RQ2-4 contributors, communication, and review",
        "local_script_or_builder": "build_repo_communication_review_patterns_notebook.py; contributor analysis scripts",
        "aggregate_outputs_used": "outputs/rq_collaboration_review_filtered_1240/",
    },
    {
        "paper_section": "RQ3 survey analysis",
        "local_script_or_builder": "py_script_analysis/analyze_vibe_coding_survey.py",
        "aggregate_outputs_used": "outputs/survey_analysis_all25_no_fdr/; outputs/survey_demographics/",
    },
])
show(provenance, 0)

,paper_section,local_script_or_builder,aggregate_outputs_used
0,Cohort construction / toy-homework filtering,paper/supplementary_data_analysis.py; py_scrip...,outputs/toy_homework_audit/*.csv
1,Data collection validation for chat sessions,"vibe-coding-scraper-main.zip: scrape.ipynb, pa...",outputs/chat_label_correction_1240/combined_se...
2,RQ1-1 AI-related repository activity,paper/supplementary_data_analysis.py; py_scrip...,outputs/repo_ai_adoption_filtered_1240/; outpu...
3,RQ1-2 chat purpose and associated commits,py_script_analysis/collect_missing_paper_label...,outputs/chat_label_correction_1240/; outputs/c...
4,RQ2-1 commit activity,paper/supplementary_data_analysis.py; relative...,outputs/relative_month_commit_activity_filtere...
5,RQ2-2 quality and maintenance signals,build_rq_quality_maintenance_monthly_notebook....,outputs/rq_quality_maintenance_filtered_1240/
6,RQ2-3 issues and pull requests,py_script_analysis/issue_pr_resolution_activit...,outputs/rq_issue_pr_filtered_1240/
7,"RQ2-4 contributors, communication, and review",build_repo_communication_review_patterns_noteb...,outputs/rq_collaboration_review_filtered_1240/
8,RQ3 survey analysis,py_script_analysis/analyze_vibe_coding_survey.py,outputs/survey_analysis_all25_no_fdr/; outputs...


## 2. Chat-Session Collection Validation

The prior paper reports 11,579 AI-assisted IDE chat sessions across 1,300 repositories and 74,998 developer messages. Its public replication package documents the expected data layout, including GitHub search results, raw Markdown chat histories, parsed chats, per-file commit histories, and repository commit histories. 

Our chat-session scraper follows the same collection structure. This notebook validates the local scraper package at a structural level: it checks that the scraper ZIP contains the parsing scripts and notebook, and that its README declares the same major raw-data directories used by the replication package.

In [4]:
scraper_zip = require("vibe-coding-scraper-main.zip")
with zipfile.ZipFile(scraper_zip) as zf:
    zip_names = zf.namelist()
    readme_name = next(n for n in zip_names if n.endswith("ReadMe.md"))
    parse_name = next(n for n in zip_names if n.endswith("parse.py"))
    parse_cli_name = next(n for n in zip_names if n.endswith("parse_cli.py"))
    scraper_readme = zf.read(readme_name).decode("utf-8", errors="ignore")
    parse_text = zf.read(parse_name).decode("utf-8", errors="ignore")
    parse_cli_text = zf.read(parse_cli_name).decode("utf-8", errors="ignore")

expected_collection_artifacts = [
    "searches/",
    "searches.json",
    "repositories.json",
    "markdowns/",
    "markdowns_cli/",
    "parsed_chats/",
    "parsed_chats_simple/",
    "parsed_chats_simple_cli/",
    "commits_path/",
    "commits/",
    "commits_history/",
    "languages.json",
]

validation_rows = []
for artifact in expected_collection_artifacts:
    validation_rows.append({
        "expected_artifact": artifact,
        "declared_in_local_scraper_readme": artifact in scraper_readme,
    })

parser_checks = pd.DataFrame([
    {"check": "SpecStory path/search marker", "present": ".specstory" in scraper_readme or ".specstory" in parse_text},
    {"check": "standard role parser: User", "present": "User" in parse_text},
    {"check": "standard role parser: Assistant/Agent", "present": ("Assistant" in parse_text) or ("Agent" in parse_text)},
    {"check": "CLI parser included", "present": len(parse_cli_text) > 1000},
])

show(pd.DataFrame(validation_rows), 0)
show(parser_checks, 0)
assert all(r["declared_in_local_scraper_readme"] for r in validation_rows)
assert parser_checks["present"].all()

,expected_artifact,declared_in_local_scraper_readme
0,searches/,True
1,searches.json,True
2,repositories.json,True
3,markdowns/,True
4,markdowns_cli/,True
5,parsed_chats/,True
6,parsed_chats_simple/,True
7,parsed_chats_simple_cli/,True
8,commits_path/,True
9,commits/,True


,check,present
0,SpecStory path/search marker,True
1,standard role parser: User,True
2,standard role parser: Assistant/Agent,True
3,CLI parser included,True


The next cell verifies the aggregate chat-session counts used in the paper after applying the strict repository filter.

In [5]:
chat_summary = read_csv("outputs/chat_label_correction_1240/combined_seven_category_summary_1240.csv")
show(chat_summary, 3)

assert int(chat_summary.loc[0, "mapped_chat_sessions"]) == 12108
assert int(chat_summary.loc[0, "mapped_repositories"]) == 1240

,mapped_chat_sessions,mapped_repositories,mapped_sessions_mean_per_repo,mapped_sessions_median_per_repo,combined_labeled_sessions_including_others,others_sessions,seven_category_labeled_sessions,seven_category_labeled_repositories,mean_distinct_seven_purposes_per_labeled_repo,median_distinct_seven_purposes_per_labeled_repo
0,12108,1240,9.765,2.0,9240,65,9175,1128,2.135,2.0


## 3. RQ1-1: AI-Related Repository Activity

This section aggregate post-adoption AI-related commit measures and file-change composition reported in the paper. AI-related commits are identified from artifact-containing commits; AI-related development activity focuses on mixed commits that also changed development files.

In [6]:
post_ai = read_csv("outputs/repo_ai_adoption_filtered_1240/repo_post_adoption_ai_commit_summary.csv")
rq1_summary = pd.DataFrame([{
    "repositories_with_post_adoption_commits": len(post_ai),
    "mean_post_adoption_commits": post_ai["post_commits"].mean(),
    "median_post_adoption_commits": post_ai["post_commits"].median(),
    "mean_ai_related_commits": post_ai["ai_related_commits"].mean(),
    "median_ai_related_commits": post_ai["ai_related_commits"].median(),
    "mean_ai_related_commit_share_pct": post_ai["ai_commit_share"].mean() * 100,
    "median_ai_related_commit_share_pct": post_ai["ai_commit_share"].median() * 100,
}])
show(rq1_summary, 2)

,repositories_with_post_adoption_commits,mean_post_adoption_commits,median_post_adoption_commits,mean_ai_related_commits,median_ai_related_commits,mean_ai_related_commit_share_pct,median_ai_related_commit_share_pct
0,1237,63.9,11.0,8.37,3.0,44.33,34.29


In [7]:
file_comp = read_csv("outputs/repo_ai_adoption_filtered_1240/ai_related_commit_file_type_distribution.csv")
size = read_csv("outputs/repo_ai_adoption_filtered_1240/post_adoption_observable_ai_commit_size.csv")

display_cols = [c for c in file_comp.columns if not SENSITIVE_COL_PAT.search(str(c))]
show(file_comp[display_cols], 2)

size_summary = pd.DataFrame([{
    "ai_related_commits": len(size),
    "mean_files_changed": size["files_changed"].mean() if "files_changed" in size else np.nan,
    "median_files_changed": size["files_changed"].median() if "files_changed" in size else np.nan,
    "mean_total_churn": size["total_churn"].mean() if "total_churn" in size else np.nan,
    "median_total_churn": size["total_churn"].median() if "total_churn" in size else np.nan,
}])
show(size_summary, 2)

,file_type,file_changes_x,share_all_ai_file_changes_pct,file_changes_y,share_development_file_changes_pct
0,source_code,99891,37.51,99891.0,43.56
1,other,47914,17.99,47914.0,20.89
2,documentation,34495,12.95,34495.0,15.04
3,generated_binary,27064,10.16,27064.0,11.80
4,configuration,7978,3.00,7978.0,3.48
5,test,7370,2.77,7370.0,3.21
6,dependency,4598,1.73,4598.0,2.01
7,ai_artifact,36982,13.89,0.0,0.00


,ai_related_commits,mean_files_changed,median_files_changed,mean_total_churn,median_total_churn
0,10355,25.72,7.0,19947.03,1789.0


In [8]:
trend = read_csv("outputs/ai_related_commit_share_extensions/ai_commit_share_by_relative_month_0_12.csv")
size_trend = read_csv("outputs/ai_related_commit_share_extensions/ai_related_commit_size_by_relative_month_0_12.csv")
show(trend.head(13), 3)
show(size_trend.head(13), 3)

,relative_month,repos,total_commits,ai_related_commits,mean_repo_month_ai_share,median_repo_month_ai_share,pooled_ai_share
0,0,995,20204,5072,0.506,0.500,0.251
1,1,398,11323,1992,0.328,0.145,0.176
2,2,240,6631,838,0.284,0.026,0.126
3,3,177,7782,492,0.189,0.000,0.063
4,4,143,3871,358,0.180,0.000,0.092
5,5,104,1927,246,0.181,0.000,0.128
6,6,91,1540,214,0.161,0.000,0.139
7,7,48,803,99,0.213,0.000,0.123
8,8,52,900,21,0.145,0.000,0.023
9,9,35,483,14,0.118,0.000,0.029


,relative_month,commits,repos,total_churn_mean,total_churn_median,total_churn_p75,total_churn_p90,source_churn_mean,source_churn_median,source_churn_p75,source_churn_p90,files_changed_mean,files_changed_median,files_changed_p75,files_changed_p90,source_files_changed_mean,source_files_changed_median,source_files_changed_p75,source_files_changed_p90
0,0,5692,996,15473.265,1638.5,6244.25,19374.2,2211.012,113.0,681.25,2752.8,23.669,7.0,18.00,52.9,9.129,2.0,7.00,20.0
1,1,2188,296,24485.203,2023.5,9751.00,41487.4,1979.533,113.0,824.50,3202.0,25.899,7.0,19.00,61.0,9.802,2.0,8.00,22.0
2,2,926,139,23536.148,1556.0,6022.75,25330.0,2900.962,53.5,508.00,2583.5,28.011,6.0,15.00,67.0,10.911,1.0,5.00,22.0
3,3,528,83,23761.947,1992.5,8566.50,34218.4,2148.513,41.0,488.75,3420.5,33.562,7.5,20.00,87.2,11.201,1.0,7.00,26.0
4,4,300,67,16635.247,1628.5,8260.00,40113.9,984.167,40.0,604.00,1980.9,26.463,7.0,18.00,56.1,9.143,1.0,6.00,18.0
5,5,250,48,22285.380,1984.5,7271.00,25701.2,1296.928,24.0,429.50,2126.9,23.420,8.0,18.75,53.1,8.296,1.0,5.00,15.0
6,6,242,40,19187.855,4014.5,15528.00,35201.7,2206.707,21.0,243.00,1100.1,22.913,5.0,11.00,40.0,7.744,1.0,3.00,11.9
7,7,95,21,32410.695,1158.0,20053.50,58297.6,1805.474,25.0,342.50,3019.2,40.316,7.0,33.50,116.8,17.337,2.0,6.50,63.2
8,8,39,25,57155.897,2249.0,19260.50,71559.8,1762.308,0.0,74.00,6738.4,72.513,19.0,93.50,300.0,19.821,1.0,12.00,69.8
9,9,30,12,11095.167,2738.5,10363.00,36477.6,3533.633,280.5,2541.00,17731.5,62.100,19.5,52.75,300.0,20.667,7.0,19.00,61.4


## 4. RQ1-2: Chat Purpose And Chat-Associated Development

Chat-purpose labels are multi-label. Therefore, category percentages are prevalence rates rather than mutually exclusive shares, and can sum above 100%.

In [9]:
purpose = read_csv("outputs/chat_label_correction_1240/corrected_multilabel_chat_purpose_distribution_1240.csv")
dominant = read_csv("outputs/chat_label_correction_1240/combined_repository_dominant_category_distribution_1240.csv")
purpose_display = purpose[[
    "Purpose", "Msg_prevalence_all_labeled_%", "Session_prevalence_%", "Sessions_with_purpose"
]].rename(columns={
    "Msg_prevalence_all_labeled_%": "message_prevalence_pct",
    "Session_prevalence_%": "session_prevalence_pct",
})
show(purpose_display, 1)
show(dominant, 1)

,Purpose,message_prevalence_pct,session_prevalence_pct,Sessions_with_purpose
0,Code Writing,34.7,69.6,6404
1,Failure Reporting,24.2,48.1,4421
2,Delegation,16.0,42.6,3915
3,Inquiry,19.3,42.8,3933
4,Context Spec.,14.1,43.0,3951
5,Workflow Control,11.5,32.4,2976
6,Validation,3.9,17.4,1601


,category,repositories,percent
0,Code Authoring,605,53.5
1,Delegation,175,15.5
2,Failure Reporting,142,12.6
3,Inquiry,100,8.8
4,Context Specification,59,5.2
5,Workflow Control,37,3.3
6,Others,8,0.7
7,Validation,5,0.4


In [10]:
assoc_overall = read_csv("outputs/chat_label_correction_1240/chat_associated_commit_recalc/chat_associated_commit_overall_summary.csv")
assoc_by_purpose = read_csv("outputs/chat_label_correction_1240/chat_associated_commit_recalc/multilabel_chat_associated_commit_by_purpose_summary.csv")
assoc_scope = read_csv("outputs/chat_label_correction_1240/chat_associated_commit_recalc/multilabel_chat_associated_commit_scope_by_chat_type.csv")
assoc_tests = read_csv("outputs/chat_label_correction_1240/chat_associated_commit_recalc/multilabel_chat_associated_commit_scope_tests.csv")

show(assoc_overall, 3)
show(assoc_by_purpose, 2)
show(assoc_scope, 2)
show(assoc_tests, 3)

,labeled_sessions_filtered,resolved_chat_associated_commit_sessions,resolved_pct,sessions_with_development_files,development_pct_of_labeled,development_pct_of_resolved,file_type_cramers_v,file_type_chi2_p,file_type_repo_blocked_perm_p
0,9240,9139,98.907,8877,96.071,97.133,0.126,0.0,1.0


,category,label_sessions,resolved_pct,development_pct_of_label_sessions,development_sessions,source_only_pct,docs_only_pct,config_only_pct,source_test_docs_config_dependency_pct
0,Code Authoring,6404,98.78,95.86,6139,29.19,15.96,4.64,7.20
1,Failure Reporting,4421,99.05,96.40,4262,29.77,15.20,7.95,6.85
2,Delegation,3915,98.57,94.25,3690,26.86,20.16,3.60,8.92
3,Inquiry,3933,99.14,94.79,3728,30.39,19.26,4.24,8.42
4,Context Specification,3951,98.79,94.13,3719,30.01,16.40,3.85,7.21
5,Workflow Control,2976,98.96,93.85,2793,29.90,17.22,4.12,10.31
6,Validation,1601,99.13,93.32,1494,27.51,14.79,6.56,10.44


,category,n_sessions,n_repos,total_churn_mean,total_churn_median,files_changed_mean,files_changed_median,file_type_categories_mean,file_type_categories_median,source_churn_mean,source_churn_median,source_files_changed_mean,source_files_changed_median
0,Code Authoring,6139,931,72543.41,989.0,52.04,7.0,2.56,2.0,3694.63,67.0,16.81,2.0
1,Failure Reporting,4262,809,21783.43,470.0,33.84,5.0,2.43,2.0,3304.67,41.0,15.51,1.0
2,Delegation,3690,830,19959.87,771.0,32.83,6.0,2.50,2.0,3205.64,43.0,13.07,1.0
3,Inquiry,3728,814,24581.62,485.0,32.59,5.0,2.43,2.0,2933.98,42.0,12.98,1.0
4,Context Specification,3719,791,56628.98,921.0,46.54,7.0,2.56,2.0,3807.41,88.0,16.37,2.0
5,Workflow Control,2793,645,20416.50,748.0,33.09,6.0,2.59,2.0,3335.90,69.0,14.84,2.0
6,Validation,1494,467,26956.55,813.0,36.41,7.0,2.67,2.0,2955.26,83.0,13.89,2.0


,metric,blocked_permutation_h,p_value,q_value
0,total_churn,133.776,0.96,1.0
1,files_changed,146.172,0.96,1.0
2,file_type_categories,58.459,0.96,1.0
3,source_churn,117.104,1.00,1.0
4,source_files_changed,131.795,1.00,1.0


## 5. RQ2-1: Commit Activity Before And After AI Adoption

The main RQ2 comparison uses the 608 post-publication repositories. Months are re-indexed relative to the AI-adoption month; the adoption month itself is excluded from pre/post comparisons.

In [11]:
commit_tests = read_csv("outputs/relative_month_commit_activity_filtered_1240/relative_month_pre_post_activity_tests_filtered.csv")
commit_its = read_csv("outputs/relative_month_commit_activity_filtered_1240/relative_month_its_pre_post_slope_models_filtered.csv")

show(commit_tests, 3)
show(commit_its[commit_its["cohort"].isin(["main_608", "older_114"])], 4)

,cohort,measure,n_repositories,pre_mean,post_mean,pre_median,post_median,mean_change,median_change,pct_increasing,pct_decreasing,p_value,effect_rank_biserial,q_value
0,main_608,total commits,270,19.343,8.134,4.844,1.000,-11.209,-2.657,22.963,74.074,0.0,-0.572,0.0
1,main_608,substantive commits,270,19.248,7.696,4.844,0.969,-11.552,-2.699,22.963,74.074,0.0,-0.584,0.0
2,main_608,source code commits,270,13.427,4.689,2.333,0.500,-8.738,-1.260,22.593,71.481,0.0,-0.601,0.0
3,older_114,total commits,109,10.380,7.312,3.875,1.125,-3.068,-1.885,27.523,72.477,0.0,-0.440,0.0
4,older_114,substantive commits,109,10.376,6.869,3.875,1.125,-3.508,-1.885,27.523,72.477,0.0,-0.454,0.0
5,older_114,source code commits,109,5.738,3.285,1.778,0.429,-2.452,-0.914,26.606,69.725,0.0,-0.525,0.0


,cohort,outcome,term,estimate_log1p,approx_percent_change,std_error,p_value,repositories,repo_months,r2,q_value
0,main_608,total commits,pre_adoption_slope,0.0093,0.9363,0.0033,0.0052,608,8888,0.4999,0.0118
1,main_608,total commits,immediate_level_change,0.0987,10.3719,0.0886,0.2652,608,8888,0.4999,0.3350
2,main_608,total commits,post_adoption_slope,-0.0109,-1.0856,0.0102,0.2832,608,8888,0.4999,0.3398
3,main_608,total commits,post_vs_pre_slope_change,-0.0202,-2.0032,0.0093,0.0290,608,8888,0.4999,0.0535
4,main_608,substantive commits,pre_adoption_slope,0.0091,0.9186,0.0033,0.0054,608,8888,0.5023,0.0118
5,main_608,substantive commits,immediate_level_change,0.0681,7.0512,0.0874,0.4356,608,8888,0.5023,0.4412
6,main_608,substantive commits,post_adoption_slope,-0.0093,-0.9234,0.0100,0.3550,608,8888,0.5023,0.4057
7,main_608,substantive commits,post_vs_pre_slope_change,-0.0184,-1.8252,0.0092,0.0444,608,8888,0.5023,0.0667
8,main_608,source-code commits,pre_adoption_slope,0.0076,0.7585,0.0033,0.0219,608,8888,0.4699,0.0439
9,main_608,source-code commits,immediate_level_change,0.0674,6.9759,0.0764,0.3776,608,8888,0.4699,0.4119


## 6. RQ2-2: Observable Code-Quality And Maintenance Signals

The following tables reproduce the defect, testing/CI, rework, bug/fix follow-up, revert, and follow-up interval summaries. The first group reports repository-period aggregate values; the ITS model outputs are kept separately for temporal trend checks.

In [12]:
defect_raw = read_csv("outputs/rq_quality_maintenance_filtered_1240/defect_recalculation/defect_raw_repo_period_tests.csv")
bug_issue_share = read_csv("outputs/rq_quality_maintenance_filtered_1240/bug_issue_share_recalc/bug_issue_share_pre_post_tests.csv")
testing_ci = read_csv("outputs/rq_quality_maintenance_filtered_1240/general_prepost_recalc/testing_ci_general_repo_period_tests_608.csv")
rework = read_csv("outputs/rq_quality_maintenance_filtered_1240/general_prepost_recalc/rework_bugfix_revert_general_repo_period_tests_608.csv")
intervals = read_csv("outputs/rq_quality_maintenance_filtered_1240/followup_interval_tests_filtered_608_114.csv")
qm_its = read_csv("outputs/rq_quality_maintenance_filtered_1240/quality_maintenance_its_recalc/quality_maintenance_relative_month_its_models_filtered_608_114.csv")

show(defect_raw, 4)
show(bug_issue_share, 4)
show(testing_ci, 4)
show(rework, 4)
show(intervals, 4)
show(qm_its.head(20), 4)

,cohort,measure,label,type,n,pre_mean,post_mean,pre_median,post_median,median_change,mean_change,pct_inc,pct_dec,pct_same,p,q,r
0,main_608,bug_fix_commits,Bug/fix commits per repository,count,608,17.3306,16.0707,0.0000,2.0000,0.0000,-1.2599,45.8882,24.8355,29.2763,0.0000,0.0000,0.2758
1,main_608,bug_fix_commit_share,Bug/fix commit share,pct,554,0.0962,0.1695,0.0000,0.1111,0.0187,0.0733,51.9856,24.1877,23.8267,0.0000,0.0000,0.4821
2,main_608,bug_issues_opened,Bug-labeled issues opened per repository,count,608,0.1513,0.0658,0.0000,0.0000,0.0000,-0.0855,0.8224,1.9737,97.2039,0.1478,0.1478,0.3987
3,older_114,bug_fix_commits,Bug/fix commits per repository,count,114,45.9298,14.0439,3.0000,2.0000,0.0000,-31.8860,31.5789,48.2456,20.1754,0.0173,0.0260,0.2871
4,older_114,bug_fix_commit_share,Bug/fix commit share,pct,111,0.1099,0.1492,0.0648,0.0952,0.0000,0.0393,48.6486,34.2342,17.1171,0.0325,0.0325,0.2567
5,older_114,bug_issues_opened,Bug-labeled issues opened per repository,count,114,0.6754,0.1579,0.0000,0.0000,0.0000,-0.5175,0.0000,7.0175,92.9825,0.0116,0.0260,1.0000


,cohort,measure,n,pre_mean,post_mean,pre_median,post_median,mean_change,median_change,pct_inc,pct_dec,pct_same,p,q
0,main_608,bug_issue_share,22,0.0547,0.0727,0.0,0.0,0.0180,0.0,18.1818,27.2727,54.5455,0.8785,0.8785
1,older_114,bug_issue_share,12,0.0762,0.0496,0.0,0.0,-0.0266,0.0,8.3333,33.3333,58.3333,0.3125,0.3125


,measure,n,pre_mean,post_mean,pre_median,post_median,median_change,mean_change,pct_inc,pct_dec,pct_same,p,r,q
0,test_commit_share,554,0.0524,0.0600,0.0000,0.0000,0.0,0.0076,21.6606,18.7726,59.5668,0.0741,0.1377,0.1481
1,test_lines_changed,608,1383.0000,3755.1332,0.0000,0.0000,0.0,2372.1332,24.3421,14.3092,61.3487,0.0000,0.3237,0.0001
2,ci_failure_rate,139,0.2911,0.2739,0.1217,0.1383,0.0,-0.0171,35.9712,39.5683,24.4604,0.6777,0.0467,0.6777
3,ci_success_rate,139,0.7089,0.7261,0.8783,0.8617,0.0,0.0171,39.5683,35.9712,24.4604,0.6777,0.0467,0.6777


,measure,n,pre_mean,post_mean,pre_median,post_median,median_change,mean_change,pct_inc,pct_dec,pct_same,p,r,q
0,rework_1d_share,413,0.5655,0.4636,0.5906,0.5000,-0.0972,-0.1019,32.2034,63.4383,4.3584,0.0000,-0.3491,0.0000
1,rework_7d_share,413,0.7240,0.5705,0.8333,0.6452,-0.1522,-0.1535,25.4237,72.6392,1.9370,0.0000,-0.5122,0.0000
2,rework_30d_share,413,0.7896,0.6309,0.9167,0.7381,-0.1311,-0.1587,20.5811,77.9661,1.4528,0.0000,-0.5687,0.0000
3,bugfix_1d_share,413,0.1970,0.1751,0.0000,0.0556,0.0000,-0.0219,31.7191,36.5617,31.7191,0.1928,0.0895,0.1928
4,bugfix_7d_share,413,0.3092,0.2372,0.1111,0.1250,0.0000,-0.0720,26.1501,44.7942,29.0557,0.0000,0.3113,0.0000
5,bugfix_30d_share,413,0.3833,0.2788,0.3014,0.1818,0.0000,-0.1045,22.0339,48.9104,29.0557,0.0000,0.4289,0.0000
6,revert_commit_share,554,0.0025,0.0055,0.0000,0.0000,0.0000,0.0029,14.2599,6.6787,79.0614,0.0001,0.4104,0.0001


,cohort,measure,label,n,pre_mean,post_mean,pre_median,post_median,median_change,mean_change,pct_longer,pct_shorter,p,r,q
0,main_608,median_same_file_rework_interval,Median time to same-file rework,357,19.1880,5.9974,0.3134,0.1549,-0.0193,-13.1907,43.6975,56.3025,0.0103,-0.1567,0.0171
1,main_608,mean_same_file_rework_interval,Mean time to same-file rework,357,29.5759,10.8221,3.4917,2.5789,-0.0060,-18.7537,49.2997,50.7003,0.0473,-0.1211,0.0591
2,main_608,median_same_file_bugfix_followup_interval,Median time to same-file bug/fix follow-up,212,50.3033,9.5984,2.8761,0.8181,-1.1023,-40.7049,26.8868,73.1132,0.0000,-0.5118,0.0000
3,main_608,mean_same_file_bugfix_followup_interval,Mean time to same-file bug/fix follow-up,212,69.3243,14.2601,8.3086,3.9030,-2.1519,-55.0642,31.6038,68.3962,0.0000,-0.4644,0.0000
4,main_608,median_revert_interval,Median explicit revert interval,3,0.0268,0.2513,0.0265,0.0049,-0.0011,0.2245,33.3333,66.6667,1.0000,0.0000,1.0000
5,older_114,median_same_file_rework_interval,Median time to same-file rework,84,62.2635,8.5346,0.9765,0.5682,-0.0205,-53.7289,45.2381,54.7619,0.3221,-0.1244,0.4027
6,older_114,mean_same_file_rework_interval,Mean time to same-file rework,84,95.8349,15.8184,23.1070,6.3872,-13.6528,-80.0165,28.5714,71.4286,0.0000,-0.6067,0.0000
7,older_114,median_same_file_bugfix_followup_interval,Median time to same-file bug/fix follow-up,53,151.6638,15.6076,12.2741,1.6497,-5.0205,-136.0562,30.1887,69.8113,0.0003,-0.5779,0.0004
8,older_114,mean_same_file_bugfix_followup_interval,Mean time to same-file bug/fix follow-up,53,218.1501,23.3353,73.3973,10.2314,-51.6383,-194.8148,16.9811,83.0189,0.0000,-0.8393,0.0000
9,older_114,median_revert_interval,Median explicit revert interval,1,0.0478,0.7477,0.0478,0.7477,0.6999,0.6999,100.0000,0.0000,1.0000,1.0000,1.0000


,cohort,measure,label,term,coef,effect,unit,p,n_obs,n_repos,q
0,main_608,test_commit_share,Testing: test-touching commit share,pretime,-0.0002,-0.0184,percentage_points,0.8498,2002,138,0.8665
1,main_608,test_commit_share,Testing: test-touching commit share,post,0.0671,6.7093,percentage_points,0.0452,2002,138,0.1374
2,main_608,test_commit_share,Testing: test-touching commit share,posttime,-0.0127,-1.2652,percentage_points,0.0502,2002,138,0.1374
3,main_608,test_commit_share,Testing: test-touching commit share,posttime_minus_pretime,-0.0125,-1.2469,percentage_points,0.0555,2002,138,0.1442
4,main_608,test_lines_changed,Testing: test lines changed,pretime,0.0028,0.2805,percent,0.4837,6206,143,0.5468
5,main_608,test_lines_changed,Testing: test lines changed,post,0.4415,55.4966,percent,0.0116,6206,143,0.0384
6,main_608,test_lines_changed,Testing: test lines changed,posttime,-0.0471,-4.5985,percent,0.0053,6206,143,0.0229
7,main_608,test_lines_changed,Testing: test lines changed,posttime_minus_pretime,-0.0499,-4.8654,percent,0.0039,6206,143,0.0185
8,main_608,ci_failure_rate,CI failure rate,pretime,-0.0015,-0.1530,percentage_points,0.8204,394,74,0.8532
9,main_608,ci_failure_rate,CI failure rate,post,0.0773,7.7272,percentage_points,0.2649,394,74,0.3709


## 7. RQ2-3: Issues And Pull Requests

Issue and PR volume is summarized over each repository's full pre/post period. Share measures distinguish issue vs PR composition among opened items and closure/merge rates among eligible items. Efficiency measures use repository-level issue resolution and PR merge times.

In [13]:
issue_pr_counts = read_csv("outputs/rq_issue_pr_filtered_1240/general_prepost_recalc/issue_pr_general_repo_period_tests_608.csv")
issue_pr_shares = read_csv("outputs/rq_issue_pr_filtered_1240/whole_period_shares/issue_pr_whole_period_share_tests_608_114.csv")
issue_pr_eff = read_csv("outputs/rq_issue_pr_filtered_1240/repo_level_issue_pr_efficiency_tests_filtered_608_114.csv")
issue_pr_its = read_csv("outputs/rq_issue_pr_filtered_1240/its_recalc/relative_month_issue_pr_its_models_filtered_608_114.csv")

show(issue_pr_counts, 4)
show(issue_pr_shares, 4)
show(issue_pr_eff, 4)
show(issue_pr_its.head(24), 4)

,measure,n,pre_mean,post_mean,pre_median,post_median,median_change,mean_change,pct_inc,pct_dec,pct_same,p,r,q
0,issues_opened,608,3.5806,0.7566,0.0,0.0,0.0,-2.8240,7.4013,7.8947,84.7039,0.353,0.1103,0.4706
1,issues_closed,608,2.6711,0.6398,0.0,0.0,0.0,-2.0312,5.0987,4.7697,90.1316,0.938,0.0115,0.9380
2,prs_opened,608,5.0559,6.3043,0.0,0.0,0.0,1.2484,29.1118,10.6908,60.1974,0.000,0.4329,0.0000
3,prs_merged,608,4.0576,3.9243,0.0,0.0,0.0,-0.1332,25.0000,8.8816,66.1184,0.000,0.4176,0.0000


,cohort,measure,label,n,pre_mean,post_mean,pre_median,post_median,mean_change,median_change,pct_inc,pct_dec,pct_same,p,q
0,main_608,issue_open_share,Issue-open share,93,0.2959,0.1916,0.0312,0.0000,-0.1043,0.0000,13.9785,40.8602,45.1613,0.0015,0.0019
1,main_608,pr_open_share,PR-open share,93,0.7041,0.8084,0.9688,1.0000,0.1043,0.0000,40.8602,13.9785,45.1613,0.0015,0.0019
2,main_608,issue_close_share,Issue close share,30,0.4005,1.0566,0.3186,1.0556,0.6561,0.3631,70.0000,10.0000,20.0000,0.0002,0.0008
3,main_608,pr_merge_share,PR merge share,77,0.7529,0.6939,0.9944,0.8205,-0.0590,0.0000,28.5714,45.4545,25.9740,0.0550,0.0550
4,older_114,issue_open_share,Issue-open share,31,0.3289,0.1980,0.2000,0.0000,-0.1309,-0.0028,12.9032,51.6129,35.4839,0.0057,0.0115
5,older_114,pr_open_share,PR-open share,31,0.6711,0.8020,0.8000,1.0000,0.1309,0.0028,51.6129,12.9032,35.4839,0.0057,0.0115
6,older_114,issue_close_share,Issue close share,13,0.7061,1.0276,0.8425,1.1111,0.3215,0.2857,69.2308,15.3846,15.3846,0.0420,0.0560
7,older_114,pr_merge_share,PR merge share,28,0.7564,0.7323,0.9324,0.7810,-0.0240,0.0000,35.7143,42.8571,21.4286,0.5480,0.5480


,cohort,outcome,repo_summary,n_repositories,pre_mean_across_repos,post_mean_across_repos,pre_median_across_repos,post_median_across_repos,median_paired_change,mean_paired_change,pct_longer,pct_shorter,p,r,q
0,main_608,issue_resolution_days,repo_mean,15,35.4269,317.6098,24.7976,70.4650,50.1698,282.1829,86.6667,13.3333,0.0006,0.9167,0.0049
1,main_608,issue_resolution_days,repo_median,15,10.4541,294.8498,6.5583,50.0697,43.5113,284.3957,80.0000,20.0000,0.0020,0.8500,0.0081
2,main_608,pr_merge_days,repo_mean,27,4.4148,4.6721,0.4388,0.1644,0.0012,0.2573,51.8519,48.1481,0.9529,0.0159,0.9799
3,main_608,pr_merge_days,repo_median,27,0.3855,1.2044,0.0227,0.0066,0.0003,0.8189,55.5556,44.4444,0.2584,0.2540,0.4134
4,older_114,issue_resolution_days,repo_mean,10,50.1892,443.2523,32.0671,87.2254,53.2106,393.0632,80.0000,20.0000,0.0137,0.8545,0.0365
5,older_114,issue_resolution_days,repo_median,10,13.8437,414.5849,6.9827,65.8106,59.8312,400.7413,70.0000,30.0000,0.0371,0.7455,0.0742
6,older_114,pr_merge_days,repo_mean,16,6.6332,7.3232,0.8658,0.7076,-0.1158,0.6900,43.7500,56.2500,0.9799,0.0147,0.9799
7,older_114,pr_merge_days,repo_median,16,0.6014,1.8940,0.0579,0.0042,0.0006,1.2926,56.2500,43.7500,0.3225,0.2941,0.4300


,cohort,measure,term,coef,effect_percent,p,n_obs,n_repos,q
0,main_608,issues_opened,pretime,-0.0010,-0.1006,0.5825,6039,270,0.8841
1,main_608,issues_opened,post,0.0082,0.8265,0.8069,6039,270,0.8841
2,main_608,issues_opened,posttime,0.0005,0.0545,0.8195,6039,270,0.8841
3,main_608,issues_opened,posttime_minus_pretime,0.0016,0.1552,0.6545,6039,270,0.8841
4,main_608,issues_closed,pretime,-0.0005,-0.0545,0.6952,6039,270,0.8841
5,main_608,issues_closed,post,0.0076,0.7592,0.7870,6039,270,0.8841
6,main_608,issues_closed,posttime,0.0005,0.0474,0.8289,6039,270,0.8841
7,main_608,issues_closed,posttime_minus_pretime,0.0010,0.1020,0.7319,6039,270,0.8841
8,main_608,prs_opened,pretime,0.0013,0.1263,0.1683,6039,270,0.8841
9,main_608,prs_opened,post,0.0101,1.0193,0.8244,6039,270,0.8841


## 8. RQ2-4: Contributor Participation, Communication, And Review

Contributor participation is measured at the repository level over full pre/post periods. Communication and review measures are also repository-level paired comparisons, with denominator-zero ratios treated as missing.

In [14]:
contributors = read_csv("outputs/rq_collaboration_review_filtered_1240/full_window_repository_contributor_tests_filtered_608_114.csv")
active_share = read_csv("outputs/rq_collaboration_review_filtered_1240/active_contributor_share/active_contributor_share_tests.csv")
comm_review = read_csv("outputs/rq_collaboration_review_filtered_1240/communication_review_paired_tests_filtered_608_114.csv")

show(contributors, 4)
show(active_share, 4)
show(comm_review, 4)

,cohort,metric,n,pre_mean,post_mean,pre_median,post_median,mean_change,median_change,pct_inc,pct_dec,p,r,q
0,main_608,active_contributor_indicator,608,0.8158,0.9737,1.0000,1.000,0.1579,0.0000,17.2697,1.4803,0.0000,0.8421,0.0000
1,main_608,unique_contributors,608,1.4441,1.6957,1.0000,1.000,0.2516,0.0000,39.3092,14.4737,0.0000,0.4542,0.0000
2,main_608,top1_commit_share,487,0.9250,0.8949,1.0000,1.000,-0.0301,0.0000,19.5072,31.8275,0.0001,-0.2779,0.0003
3,main_608,contributor_hhi,487,0.8993,0.8593,1.0000,1.000,-0.0400,0.0000,19.5072,31.8275,0.0001,-0.2896,0.0002
4,main_608,commits_per_contributor,487,37.9021,33.4269,8.5000,11.000,-4.4752,1.0000,53.1828,42.7105,0.1898,0.0700,0.2531
5,main_608,churn_per_contributor,487,109960.5655,198010.2433,9265.0000,22360.500,88049.6778,8021.0000,69.4045,30.5955,0.0000,0.4363,0.0000
6,main_608,files_per_contributor,487,353.3013,390.5689,68.0000,104.500,37.2677,22.0000,60.3696,39.2197,0.0002,0.1962,0.0003
7,main_608,substantive_commits,608,65.6349,59.3964,6.0000,15.000,-6.2385,4.0000,61.0197,35.0329,0.0000,0.2011,0.0001
8,older_114,active_contributor_indicator,114,0.9737,0.9737,1.0000,1.000,0.0000,0.0000,2.6316,2.6316,1.0000,0.0000,1.0000
9,older_114,unique_contributors,114,2.9211,1.7456,1.0000,1.000,-1.1754,0.0000,18.4211,35.0877,0.0264,-0.3199,0.0384


,cohort,measure,label,n,pre_mean,post_mean,pre_median,post_median,mean_change,median_change,pct_inc,pct_dec,pct_same,p,q
0,main_608,active_contributor_share,Active contributor share,605,0.7892,0.8678,1.0,1.0,0.0787,0.0,26.4463,17.1901,56.3636,0.000,0.000
1,main_608,any_active_contributor,Repository has any active contributor,608,0.9030,0.9753,1.0,1.0,0.0724,0.0,9.2105,1.9737,88.8158,0.000,0.000
2,older_114,active_contributor_share,Active contributor share,114,0.8786,0.7409,1.0,1.0,-0.1377,0.0,15.7895,35.9649,48.2456,0.002,0.004
3,older_114,any_active_contributor,Repository has any active contributor,114,0.9737,0.9737,1.0,1.0,0.0000,0.0,2.6316,2.6316,94.7368,1.000,1.000


,cohort,measure,label,measure_type,n_repositories_included,n_repositories_excluded_missing_pre_or_post,pre_mean,post_mean,pre_median,post_median,median_paired_change,mean_paired_change,median_percent_change_when_pre_positive,median_log_ratio,pct_repositories_increasing,pct_repositories_decreasing,p_value,effect_rank_biserial,q_value
0,main_608,comments_per_issue_pr,Comments per issue/PR,ratio,90,518,1.4365,2.0101,0.7926,1.4367,0.0000,0.5736,0.0000,0.0000,43.3333,30.0000,0.0142,0.3474,0.0991
1,main_608,pr_review_comments_per_pr,PR review comments per PR,ratio,74,534,1.6768,2.0564,0.0000,0.0000,0.0000,0.3796,-0.5241,0.0000,28.3784,17.5676,0.2416,0.2303,0.4227
2,main_608,unique_commenters,Unique row-level issue commenters,count,608,0,0.7303,0.2928,0.0000,0.0000,0.0000,-0.4375,-0.6808,0.0000,7.8947,5.2632,0.1887,0.1639,0.4227
3,main_608,unique_reviewers,Unique PR reviewers,count,608,0,0.1069,0.0905,0.0000,0.0000,0.0000,-0.0164,-1.0000,0.0000,4.4408,3.4539,0.4613,0.1139,0.6458
4,main_608,top_commenter_share,Top issue-commenter share,share,27,581,0.7440,0.7550,0.6970,0.8571,0.0000,0.0111,0.0000,0.0000,44.4444,33.3333,0.7676,0.0736,0.7676
5,main_608,commenter_hhi,Issue-commenter HHI,hhi,27,581,0.6618,0.7104,0.5776,0.7551,0.0432,0.0485,0.0832,0.0377,51.8519,29.6296,0.2296,0.2925,0.4227
6,main_608,reviewer_hhi,Reviewer HHI,hhi,12,596,0.8138,0.7711,1.0000,1.0000,0.0000,-0.0428,0.0000,0.0000,16.6667,33.3333,0.5625,-0.3333,0.6562
7,older_114,comments_per_issue_pr,Comments per issue/PR,ratio,29,85,1.3445,1.9482,1.0417,0.5714,0.0000,0.6037,-0.2633,0.0000,27.5862,48.2759,0.9870,0.0040,0.9870
8,older_114,pr_review_comments_per_pr,PR review comments per PR,ratio,26,88,0.0639,1.3331,0.0000,0.0000,0.0000,1.2691,-1.0000,0.0000,15.3846,11.5385,0.1763,0.5714,0.4897
9,older_114,unique_commenters,Unique row-level issue commenters,count,114,0,3.5965,0.7544,0.0000,0.0000,0.0000,-2.8421,-0.9746,0.0000,2.6316,16.6667,0.0016,-0.7628,0.0111


## 9. RQ3: Survey Findings

This section uses only aggregate survey outputs.

In [ ]:
constructs = read_csv("outputs/survey_analysis_all25_no_fdr/construct_summary_all25_no_fdr.csv")
self_vs_others = read_csv("outputs/survey_analysis_all25_no_fdr/self_vs_others_all25_no_fdr.csv")
appropriate = read_csv("outputs/survey_analysis_all25_no_fdr/Q17_multiselect_exact_counts_all25.csv")
avoid = read_csv("outputs/survey_analysis_all25_no_fdr/Q18_multiselect_exact_counts_all25.csv")
demo = read_csv("outputs/survey_demographics/all25_demographic_distribution_grouped.csv")

show(constructs, 4)
show(self_vs_others, 4)
show(appropriate, 2)
show(avoid, 2)
show(demo, 2)